# VisionBridge — Colab training + prediction smoke test

This notebook changes **notebook code only**. It uses the existing VisionBridge backend/training pipeline without modifying repository Python files.

Pipeline: existing processed keypoints → dataset validation → one-batch CTC sanity check → resumable base-model training → real predictions → CER/exact-match report.

Feature contract: pose=132, face=1404. Sequences are capped at MAX_SEQUENCE_LENGTH=1024 by the repository dataset collator.


## 1. Install only what is missing

Do **not** uninstall TensorFlow/MediaPipe/Protobuf and do not reinstall PyTorch. This avoids the long dependency churn that was hanging the previous notebook.


In [ ]:
import importlib.util, subprocess, sys

required = {
    'numpy': 'numpy==1.26.4',
    'google.protobuf': 'protobuf==4.25.9',
    'mediapipe': 'mediapipe==0.10.21',
    'cv2': 'opencv-python-headless',
    'pandas': 'pandas',
    'kagglehub': 'kagglehub',
}
missing = [spec for module, spec in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing missing packages:', missing)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', *missing], check=True)
    print('Packages installed. If MediaPipe/NumPy/Protobuf were installed, restart the runtime once, then continue.')
else:
    print('All required packages are already installed; no reinstall needed.')


## 2. Verify runtime

If the previous cell installed packages for the first time, restart the Colab runtime before running this cell.


In [ ]:
import torch, numpy, google.protobuf, mediapipe as mp
print('NumPy:', numpy.__version__)
print('Protobuf:', google.protobuf.__version__)
print('MediaPipe:', mp.__version__)
print('Holistic API:', hasattr(mp, 'solutions'))
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
assert hasattr(mp, 'solutions'), 'MediaPipe Holistic is unavailable. Restart the runtime and rerun this cell.'


## 3. Load a fresh VisionBridge checkout

The notebook always uses the current `main` checkout so stale Colab code does not silently get used.


In [ ]:
import os, shutil, subprocess
BASE='/content'
REPO_ROOT=f'{BASE}/VisionBridge'
if os.path.isdir(REPO_ROOT):
    subprocess.run(['git','-C',REPO_ROOT,'pull','--ff-only'], check=True)
else:
    subprocess.run(['git','clone','https://github.com/BharathWaj-K-R/VisionBridge.git',REPO_ROOT], check=True)
os.chdir(REPO_ROOT)
print('Repository:', REPO_ROOT)

with open('backend/app/models/base_model.py', encoding='utf-8') as f:
    model_src=f.read()
with open('backend/scripts/extract_keypoints.py', encoding='utf-8') as f:
    extract_src=f.read()
assert 'POSE_INPUT_DIM = 33 * 4' in model_src and 'FACE_INPUT_DIM = 468 * 3' in model_src
assert 'POSE_FEATURE_DIM = 33 * 4' in extract_src and 'FACE_FEATURE_DIM = 468 * 3' in extract_src
assert '1434' not in model_src and '1434' not in extract_src
print('132/1404 feature contract verified.')


## 4. Use existing processed data if present

This is the important fast path: if `data/processed/isltranslate` already contains the 687 extracted samples, **do not download videos or rerun MediaPipe extraction**.

If processed data is absent, the cell downloads the Kaggle ISL-CSLTR dataset and prepares the raw-video extraction path.


In [ ]:
from pathlib import Path
import glob, pandas as pd

PROCESSED=Path('data/processed/isltranslate')
processed_csv=PROCESSED/'ISLTranslate.csv'
pose_count=len(list((PROCESSED/'pose').glob('*.npy'))) if (PROCESSED/'pose').exists() else 0
face_count=len(list((PROCESSED/'face').glob('*.npy'))) if (PROCESSED/'face').exists() else 0

if processed_csv.exists() and pose_count>0 and face_count>0:
    print(f'Using existing processed dataset: {processed_csv}')
    print(f'Pose files: {pose_count}, face files: {face_count}')
else:
    import kagglehub
    dataset_path=kagglehub.dataset_download('drblack00/isl-csltr-indian-sign-language-dataset')
    print('Dataset:', dataset_path)
    candidates=[d for d in glob.glob(os.path.join(dataset_path,'**','*Sentence_Level*'),recursive=True) if os.path.isdir(d) and 'Video' in os.path.basename(d)]
    assert len(candidates)==1, f'Expected one sentence-video directory, found: {candidates}'
    video_root=candidates[0]
    raw=Path('data/raw_videos'); labels=Path('data/labels/ISLTranslate.csv')
    raw.mkdir(parents=True,exist_ok=True); labels.parent.mkdir(parents=True,exist_ok=True)
    videos=[]
    for ext in ('*.mp4','*.MP4','*.avi','*.AVI','*.mov','*.MOV'):
        videos.extend(glob.glob(os.path.join(video_root,'**',ext),recursive=True))
    assert videos, f'No videos found under {video_root}'
    rows=[]
    for i,v in enumerate(sorted(videos)):
        uid=f'clip{i:04d}'
        text=Path(v).parent.name.replace('_',' ').strip()
        if not text: continue
        dst=raw/f'{uid}.mp4'
        if not dst.exists(): dst.symlink_to(Path(v))
        rows.append({'uid':uid,'text':text})
    pd.DataFrame(rows).to_csv(labels,index=False)
    print(f'Prepared {len(rows)} video labels. Run the extraction command in the next cell.')


## 5. Extract only when necessary


In [ ]:
from pathlib import Path
if not (Path('data/processed/isltranslate/ISLTranslate.csv').exists() and list(Path('data/processed/isltranslate/pose').glob('*.npy'))):
    !python backend/scripts/extract_keypoints.py --videos_dir data/raw_videos --labels_csv data/labels/ISLTranslate.csv --out_dir data/processed/isltranslate
else:
    print('Extraction skipped — processed keypoints already exist.')


## 6. Validate dataset + model before training


In [ ]:
import sys, torch
sys.path.insert(0,'backend')
from torch.utils.data import DataLoader
from app.training.isltranslate import ISLTranslateKeypointDataset, SimpleCharTokenizer, collate_ctc_batch
from app.models.base_model import VisionBridgeBaseModel, POSE_INPUT_DIM, FACE_INPUT_DIM, MAX_SEQUENCE_LENGTH

tok=SimpleCharTokenizer()
ds=ISLTranslateKeypointDataset('data/processed/isltranslate', tokenizer=tok)
loader=DataLoader(ds,batch_size=min(2,len(ds)),shuffle=True,collate_fn=collate_ctc_batch)
batch=next(iter(loader))
print('Examples:',len(ds))
print('Pose:',tuple(batch['pose'].shape),'Face:',tuple(batch['face'].shape))
print('Input lengths:',batch['input_lengths'].tolist())
assert batch['pose'].shape[-1]==POSE_INPUT_DIM==132
assert batch['face'].shape[-1]==FACE_INPUT_DIM==1404
assert batch['pose'].shape[1]<=MAX_SEQUENCE_LENGTH
model=VisionBridgeBaseModel(vocab_size=tok.vocab_size).eval()
with torch.no_grad(): logits=model(batch['pose'],batch['face'])
loss=torch.nn.CTCLoss(blank=0,zero_infinity=True)(torch.log_softmax(logits,dim=-1).transpose(0,1),batch['labels'],batch['input_lengths'],batch['label_lengths'])
print('Logits:',tuple(logits.shape))
print('CTC sanity loss:',float(loss))
assert torch.isfinite(loss)
print('SANITY PASS — safe to train.')


## 7. Train/resume the base model

The repository training script saves the best validation weights and can resume using a full checkpoint.


In [ ]:
%cd /content/VisionBridge
!PYTHONPATH=backend python -m app.training.train_base_model --data-dir data/processed/isltranslate --output backend/app/models/weights/base_model.pt --epochs 15 --batch-size 4 --checkpoint-dir runs/base_model --resume --device cuda


## 8. Evaluate with the exact live decoder


In [ ]:
%cd /content/VisionBridge
!PYTHONPATH=backend python -m app.training.evaluate --data-dir data/processed/isltranslate --weights backend/app/models/weights/base_model.pt --max-samples 100 --worst-n 10 --device cuda


## 9. Real prediction check — ground truth vs model output

This uses the same `decode_logits()` function used by the backend inference service, so this is the meaningful check that the trained model actually produces text rather than merely producing finite logits.


In [ ]:
import sys, torch
sys.path.insert(0,'backend')
from app.models.base_model import load_frozen_base_model
from app.services.inference_service import decode_logits
from app.training.isltranslate import ISLTranslateKeypointDataset, SimpleCharTokenizer, _downsample_to_max_length

weights='backend/app/models/weights/base_model.pt'
vocab='backend/app/models/weights/base_model.vocab.json'
tok=SimpleCharTokenizer.load(__import__('pathlib').Path(vocab))
ds=ISLTranslateKeypointDataset('data/processed/isltranslate',tokenizer=tok)
model=load_frozen_base_model(weights,vocab_size=tok.vocab_size).to('cuda' if torch.cuda.is_available() else 'cpu')
device=next(model.parameters()).device

for i in [0, len(ds)//4, len(ds)//2, (3*len(ds))//4, len(ds)-1]:
    item=ds[i]
    pose,face=_downsample_to_max_length(item['pose'],item['face'],item['uid'])
    with torch.no_grad():
        logits=model(pose.unsqueeze(0).to(device),face.unsqueeze(0).to(device))
    pred,conf=decode_logits(logits)
    print('---',item['uid'])
    print('GROUND TRUTH:',item['text'])
    print('PREDICTED:   ',pred)
    print('CONFIDENCE:  ',round(conf,4))

print('Prediction smoke test complete.')


## 10. Download the verified artifacts


In [ ]:
from google.colab import files
files.download('backend/app/models/weights/base_model.pt')
files.download('backend/app/models/weights/base_model.vocab.json')
